# FEVER Closed-Book with Ollama (Colab)

This notebook runs **closed-book** fact verification on the FEVER dataset using a local **Ollama** model in Colab.

**Steps**
1. Install & start Ollama.
2. Pull a small LLM (e.g., `qwen2:7b`).
3. Load FEVER (upload JSONL files or read from Drive).
4. Prompt claims directly (no retrieval).
5. Evaluate accuracy & confusion matrix; save results.


## 1) Runtime & setup
- **Runtime → Change runtime type → GPU** (T4/L4/A100).
- Run the following cell to install and start Ollama, and to install Python deps.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama server in the background
!nohup ollama serve > /dev/null 2>&1 &

# Python client + utilities
!pip -q install ollama tqdm pandas numpy
!nvidia-smi


print('Ollama installed. If you see connection errors later, re-run the serve command above.')

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Wed Nov 12 16:20:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |


## 2) Pull a small model
Pick one. Defaults below to **Qwen2 7B**. You may switch to `gemma:7b` or `llama3.1:8b`.

In [ ]:
!ollama pull qwen2:7b
# Alternative models:
# !ollama pull gemma:7b
# !ollama pull llama3.1:8b


## 3) Bring FEVER into the notebook
Upload `train.jsonl`, `dev.jsonl`, and/or `test.jsonl` to `/content` **or** mount Google Drive and use your paths below.

In [ ]:
# Optional: mount Google Drive if your FEVER files live there
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted:', e)

from google.colab import files
up = files.upload()

# Set FEVER file paths here. Update as needed.
# DEV_PATH = '/content/MyDrive/fever/shared_task_dev.jsonl'
DEV_PATH = '/content/train.jsonl'

Mounted at /content/drive


Saving train.jsonl to train.jsonl


## 4) Minimal FEVER loader
We normalize FEVER labels to `SUPPORTS`, `REFUTES`, `NOT_ENOUGH_INFO`.

In [ ]:
import json
from typing import List, Dict

def read_fever_jsonl(path: str) -> List[Dict]:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def norm_gold_label(lbl: str) -> str:
    return lbl.replace(' ', '_').upper()

dev_rows = read_fever_jsonl(DEV_PATH)
print('Loaded dev rows:', len(dev_rows))
print('Keys:', list(dev_rows[0].keys()))

Loaded dev rows: 145449
Keys: ['id', 'verifiable', 'label', 'claim', 'evidence']


## 5) Closed-book prompt + Ollama call
We force a single-label output and reprompt once if malformed.

In [ ]:
import re, ollama, time
from tqdm import tqdm

LABELS = {"SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"}
LABEL_RE = re.compile(r'^\s*(SUPPORTS|REFUTES|NOT[_ ]ENOUGH[_ ]INFO)\s*$', re.I)

CLOSED_BOOK_TMPL = (
    "You are a strict fact verifier.\n"
    "Without external sources, decide if the claim is true, false, or unknown.\n"
    "If you do not know for sure, output NOT_ENOUGH_INFO.\n\n"
    "Claim: \"{claim}\"\n\n"
    "Answer with exactly ONE of these labels on a single line:\n"
    "SUPPORTS\nREFUTES\nNOT_ENOUGH_INFO\n\n"
    "Answer:\n"
)

def normalize_label(text: str):
    m = LABEL_RE.search(text or "")
    if not m:
        return None
    lab = m.group(1).upper().replace(' ', '_')
    if lab == 'NOT_ENOUGH_INFO' or lab in {'SUPPORTS', 'REFUTES'}:
        return lab
    return None

def ask_ollama(model: str, prompt: str, temperature: float = 0.1, retry: int = 1) -> str:
    for _ in range(retry + 1):
        r = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}],
                        options={"temperature": temperature})
        out = (r["message"]["content"] or "").strip()
        lab = normalize_label(out)
        if lab:
            return lab
        time.sleep(0.2)
    return "NOT_ENOUGH_INFO"  # safe fallback


## 6) Run inference on a split (closed-book)
Set `max_n` to a small number (e.g., 200) for a quick sanity check before full runs.

In [ ]:
import pandas as pd

def run_closed_book(rows, model="qwen2:7b", max_n=None):
    preds, golds, claim_ids, claims = [], [], [], []
    use_rows = rows if max_n is None else rows[:max_n]
    for r in tqdm(use_rows):
        claim = r["claim"]
        gold = norm_gold_label(r.get("label", "NOT ENOUGH INFO"))
        prompt = CLOSED_BOOK_TMPL.format(claim=claim)
        pred = ask_ollama(model, prompt, temperature=0.1, retry=1)
        preds.append(pred)
        golds.append(gold)
        claim_ids.append(r.get("id", None))
        claims.append(claim)
    df = pd.DataFrame({"id": claim_ids, "claim": claims, "gold": golds, "pred": preds})
    return df

# Quick run first (adjust max_n or set to None for full dev)
df_dev = run_closed_book(dev_rows, model="qwen2:7b", max_n=None)
df_dev.head()

100%|██████████| 145449/145449 [8:20:10<00:00,  4.85it/s]


,id,claim,gold,pred
0,75397,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,NOT_ENOUGH_INFO
1,150448,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS
2,214861,"History of art includes architecture, dance, s...",SUPPORTS,SUPPORTS
3,156709,Adrienne Bailon is an accountant.,REFUTES,NOT_ENOUGH_INFO
4,83235,System of a Down briefly disbanded in limbo.,NOT_ENOUGH_INFO,SUPPORTS


## 7) Metrics (accuracy + confusion matrix)

In [ ]:
import numpy as np

ORDER = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"]
idx = {l:i for i,l in enumerate(ORDER)}

def accuracy(gold, pred):
    return float(np.mean([g==p for g,p in zip(gold, pred)]))

def confusion(gold, pred):
    M = np.zeros((3,3), dtype=int)
    for g,p in zip(gold, pred):
        M[idx[g], idx[p]] += 1
    import pandas as pd
    return pd.DataFrame(M, index=[f"gold:{l}" for l in ORDER], columns=[f"pred:{l}" for l in ORDER])

acc = accuracy(df_dev["gold"].tolist(), df_dev["pred"].tolist())
cm  = confusion(df_dev["gold"].tolist(), df_dev["pred"].tolist())
print('Accuracy:', acc)
cm

Accuracy: 0.59980474255581


,pred:SUPPORTS,pred:REFUTES,pred:NOT_ENOUGH_INFO
gold:SUPPORTS,54931,6245,18859
gold:REFUTES,3564,10035,16176
gold:NOT_ENOUGH_INFO,8785,4579,22275


## 8) Save results
Change the output path as needed.

In [ ]:
out_path = "/content/fever_closedbook_qwen2_7b_dev_subset.csv"
df_dev.to_csv(out_path, index=False)
out_path

'/content/fever_closedbook_qwen2_7b_dev_subset.csv'

## Notes & Tips
- If a 7B model OOMs, switch to a smaller or quantized variant.
- Keep `temperature` low (0.0–0.2) and the strict label regex guard.
- FEVER uses `NOT ENOUGH INFO` (spaces); we normalize to `NOT_ENOUGH_INFO`.
- For full runs, set `max_n=None` when calling `run_closed_book`.
- Re-run the `ollama serve` command if connection errors.